# Barcelona Data Preparation

This notebook prepares the Barcelona AirDNA/Airbnb dataset for the capstone project.

## Final outputs

- `barcelona_listings_clean.csv`  
  - Unit of analysis: **1 row = 1 Airbnb listing**
- `barcelona_monthly_metrics_clean.csv`  
  - Unit of analysis: **1 row = 1 Airbnb listing + 1 month**

The monthly dataset is extracted from the embedded JSON stored in the `months` column of the listings dataset.


## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

## 2. Define project paths

The notebook assumes it is executed from the `notebooks/` folder.

Expected structure:

```text
KPMG_Airbnb_Capstone/
├── data/
│   ├── raw/
│   │   └── barcelona/
│   └── processed/
│       └── barcelona/
└── notebooks/
```


In [ ]:
BASE_DIR = Path("..")

RAW_DIR = BASE_DIR / "data" / "raw" / "barcelona"
PROCESSED_DIR = BASE_DIR / "data" / "processed" / "barcelona"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Raw directory:", RAW_DIR.resolve())
print("Processed directory:", PROCESSED_DIR.resolve())

## 3. Load raw Barcelona listings dataset

This notebook uses the listings dataset as the main raw source because it contains:

- one row per listing,
- listing and host characteristics,
- location variables,
- performance metrics,
- the embedded monthly history in the `months` column.

If your local file has `(1)` in the name, rename it to:

```text
listings_BARCELONA_CONVERT_FROM_PARQUET.csv
```


In [ ]:
listings_file = RAW_DIR / "listings_BARCELONA_CONVERT_FROM_PARQUET.csv"

if not listings_file.exists():
    raise FileNotFoundError(
        f"File not found: {listings_file}\n"
        "Please check that the file exists in data/raw/barcelona/ "
        "and rename it to listings_BARCELONA_CONVERT_FROM_PARQUET.csv"
    )

listings = pd.read_csv(listings_file)

print("Listings shape:", listings.shape)

In [ ]:
listings.head()

## 4. Initial data audit

Before cleaning, we confirm the unit of analysis and inspect the main structure of the dataset.


In [ ]:
print("Rows:", len(listings))
print("Unique listing_id:", listings["listing_id"].nunique())
print("Duplicate listing_id:", listings["listing_id"].duplicated().sum())

Expected interpretation:

- If `Rows` equals `Unique listing_id`
- And `Duplicate listing_id` equals `0`

then **1 row = 1 unique Airbnb listing**.


In [ ]:
listings.dtypes.value_counts()

In [ ]:
missing = (
    listings.isnull()
    .sum()
    .sort_values(ascending=False)
)

missing.head(30)

In [ ]:
missing_pct = (
    listings.isnull()
    .mean()
    .sort_values(ascending=False)
    * 100
)

missing_pct.head(30)

In [ ]:
column_audit = pd.DataFrame({
    "column": listings.columns,
    "dtype": listings.dtypes.values,
    "missing_pct": listings.isnull().mean().values * 100
}).sort_values("missing_pct", ascending=False)

column_audit.head(30)

## 5. Geographic variables

For Barcelona:

- `neighborhood` corresponds to the broader district level.
- `subdivision` corresponds to the more granular neighbourhood/barrio level.

For this project, `subdivision` should be the main geographic variable for neighbourhood-level analysis, while `neighborhood` is useful for higher-level aggregation.


In [ ]:
print("Neighbourhood-like columns:")
print([c for c in listings.columns if "neigh" in c.lower()])

print("\nSubdivision-like columns:")
print([c for c in listings.columns if "sub" in c.lower()])

In [ ]:
listings["neighborhood"].value_counts(dropna=False).head(20)

In [ ]:
listings["subdivision"].value_counts(dropna=False).head(20)

## 6. Host audit

This is useful for measuring commercialisation and host concentration.


In [ ]:
print("Listings:", listings["listing_id"].nunique())
print("Unique hosts:", listings["host_id"].nunique())

host_listing_counts = (
    listings.groupby("host_id")["listing_id"]
    .nunique()
    .sort_values(ascending=False)
)

host_listing_counts.describe()

In [ ]:
host_listing_counts.head(20)

## 7. Select core listing-level columns

The goal is to keep only columns that are useful for the capstone:

- listing identification,
- host identification,
- location,
- property characteristics,
- review/rating information,
- compliance/licensing,
- recent performance metrics,
- embedded monthly history.


In [ ]:
core_columns = [
    "listing_id",
    "host_id",

    "professional_management",
    "superhost",
    "cohost",

    "neighborhood",
    "subdivision",

    "latitude",
    "longitude",

    "listing_type",
    "room_type",

    "guests",
    "bedrooms",
    "beds",
    "baths",

    "num_reviews",
    "star_rating",

    "registration",

    "ttm_revenue",
    "ttm_days_booked",
    "ttm_avg_rate",

    "l90d_revenue",
    "l90d_occupancy",
    "l90d_revpar",

    "months"
]

missing_cols = [c for c in core_columns if c not in listings.columns]
print("Missing selected columns:", missing_cols)

In [ ]:
if missing_cols:
    raise ValueError(f"The following selected columns are missing: {missing_cols}")

listings_clean = listings[core_columns].copy()

print("Listings clean shape:", listings_clean.shape)
listings_clean.head()

## 8. Create listing-level features

These features will be useful for the risk score, clustering and policy analysis.


In [ ]:
# Count how many listings each host manages
host_listing_count = listings_clean.groupby("host_id")["listing_id"].transform("nunique")

listings_clean["host_listing_count"] = host_listing_count
listings_clean["multi_listing_host"] = listings_clean["host_listing_count"] > 1
listings_clean["host_5_plus_listings"] = listings_clean["host_listing_count"] >= 5
listings_clean["host_10_plus_listings"] = listings_clean["host_listing_count"] >= 10

# Entire-home flag
listings_clean["entire_home_flag"] = (
    listings_clean["room_type"]
    .astype(str)
    .str.lower()
    .str.contains("entire|whole|apartment|home", regex=True, na=False)
)

# Professional management flag.
# Keep original value, but create a boolean flag where available.
listings_clean["professional_management_flag"] = (
    listings_clean["professional_management"]
    .fillna(False)
    .astype(bool)
)

# Registration / licence availability flag.
listings_clean["has_registration"] = listings_clean["registration"].notna()

listings_clean.head()

## 9. Export listing-level clean dataset

In [ ]:
listings_clean.to_csv(
    PROCESSED_DIR / "barcelona_listings_clean.csv",
    index=False
)

print("Saved:", PROCESSED_DIR / "barcelona_listings_clean.csv")

## 10. Inspect embedded monthly history

The `months` column is a JSON string. Each row contains a list of monthly performance records for that listing.


In [ ]:
sample_months = json.loads(listings.loc[0, "months"])

print("Type:", type(sample_months))
print("Number of months in first listing:", len(sample_months))
sample_months[0]

In [ ]:
sample_df = pd.DataFrame(sample_months)

sample_df["month_date"] = pd.to_datetime(
    sample_df["date"],
    unit="ms"
)

sample_df.head()

## 11. Expand monthly history for all listings

This transforms the embedded `months` JSON into a proper monthly panel dataset.

Final unit of analysis:

```text
1 row = 1 Airbnb listing + 1 month
```


In [ ]:
monthly_records = []

for _, row in listings.iterrows():
    listing_id = row["listing_id"]

    if pd.isna(row["months"]):
        continue

    months_data = json.loads(row["months"])

    for month in months_data:
        month_record = month.copy()
        month_record["listing_id"] = listing_id
        monthly_records.append(month_record)

monthly_df = pd.DataFrame(monthly_records)

monthly_df["month_date"] = pd.to_datetime(
    monthly_df["date"],
    unit="ms"
)

print("Monthly dataset shape:", monthly_df.shape)
monthly_df.head()

## 12. Monthly dataset audit

In [ ]:
print("Rows:", len(monthly_df))
print("Unique listings:", monthly_df["listing_id"].nunique())
print("Min month:", monthly_df["month_date"].min())
print("Max month:", monthly_df["month_date"].max())

In [ ]:
monthly_df.info()

In [ ]:
monthly_df.isnull().sum().sort_values(ascending=False)

## 13. Clean monthly metrics

Rename the main pricing and revenue columns for clarity:

- `rate_avg` → `avg_daily_rate`
- `rev_par` → `revpar`


In [ ]:
monthly_clean = monthly_df.rename(
    columns={
        "rate_avg": "avg_daily_rate",
        "rev_par": "revpar"
    }
)

monthly_clean.head()

## 14. Export monthly clean dataset

In [ ]:
monthly_clean.to_csv(
    PROCESSED_DIR / "barcelona_monthly_metrics_clean.csv",
    index=False
)

print("Saved:", PROCESSED_DIR / "barcelona_monthly_metrics_clean.csv")

## 15. Final outputs summary

This notebook creates two processed Barcelona datasets:

### `barcelona_listings_clean.csv`

Unit of analysis:

```text
1 row = 1 Airbnb listing
```

Use for:

- STR density,
- entire-home share,
- host concentration,
- professional management analysis,
- listing-level risk features,
- clustering inputs.

### `barcelona_monthly_metrics_clean.csv`

Unit of analysis:

```text
1 row = 1 Airbnb listing + 1 month
```

Coverage:

```text
March 2021 to February 2026
```

Use for:

- monthly occupancy,
- average daily rate,
- revenue,
- RevPAR,
- historical trends,
- emerging hotspot detection.
